<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/Hierarchal_App_Market_Visualization_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
import re
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore.csv"

# Load data (falls back to local file if needed)
try:
  raw_df = pd.read_csv(DATA_URL)
except Exception:
  raw_df = pd.read_csv("googleplaystore.csv")

df = raw_df.copy()


# ---------------------------------------------------------
# 2. Data Cleaning & Type Formatting
# ---------------------------------------------------------
# Convert Rating and Reviews to numeric
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")

# Clean and convert Installs
df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")


# Convert Size to Megabytes (MB)
def parse_size_mb(size_val):
  size_str = str(size_val).strip()
  if size_str.endswith("M"):
    return float(size_str[:-1])
  elif size_str.endswith("k"):
    return float(size_str[:-1]) / 1024.0
  return np.nan


df["Size_MB"] = df["Size"].apply(parse_size_mb)

# Ensure App Type is clean
df["App_Type"] = df["Type"].fillna("Free").astype(str).str.strip()

# Top-level country definition (defaults to 'Global' if Country column is not in raw data)
if "Country" not in df.columns:
  df["Country"] = "Global"

# Remove duplicate apps within the same category
df = df.drop_duplicates(subset=["App", "Category"]).dropna(
    subset=["Rating", "Reviews", "Installs", "Size_MB"]
)

# ---------------------------------------------------------
# 3. Filtering Criteria
# ---------------------------------------------------------
# Filter 1: App names containing no numbers
has_no_digits = ~df["App"].str.contains(r"\d", regex=True, na=False)

# Filter 2: Metrics bounds (Rating >= 4.0, Installs > 10,000, Reviews > 1,000, Size between 15 MB and 80 MB)
meets_metrics = (
    (df["Rating"] >= 4.0)
    & (df["Installs"] > 10000)
    & (df["Reviews"] > 1000)
    & (df["Size_MB"] >= 15.0)
    & (df["Size_MB"] <= 80.0)
)

# Filter 3: Exclude categories beginning with 'A', 'C', 'G', or 'S'
is_eligible_cat = ~df["Category"].str.upper().str.startswith(
    ("A", "C", "G", "S")
)

filtered_df = df[has_no_digits & meets_metrics & is_eligible_cat].copy()

# Filter 4: Top 5 eligible categories by total installs
top_5_categories = (
    filtered_df.groupby("Category")["Installs"].sum().nlargest(5).index.tolist()
)
filtered_df = filtered_df[filtered_df["Category"].isin(top_5_categories)].copy()

# ---------------------------------------------------------
# 4. Translations & Rating Bands
# ---------------------------------------------------------
category_translations = {
    "BUSINESS": "வணிகம்",  # Tamil
    "TRAVEL_AND_LOCAL": "Voyages et transport local",  # French
    "PRODUCTIVITY": "Productividad",  # Spanish
}
filtered_df["Display_Category"] = (
    filtered_df["Category"].replace(category_translations).str.strip()
)


# Assign Rating Bands
def assign_rating_band(r):
  if 4.0 <= r < 4.2:
    return "4.0–4.2"
  elif 4.2 <= r < 4.5:
    return "4.2–4.5"
  elif 4.5 <= r < 4.7:
    return "4.5–4.7"
  elif 4.7 <= r <= 5.0:
    return "4.7–5.0"
  return np.nan


filtered_df["Rating_Band"] = filtered_df["Rating"].apply(assign_rating_band)
filtered_df = filtered_df.dropna(subset=["Rating_Band"]).copy()

# ---------------------------------------------------------
# 5. Build Sunburst Tree Hierarchy
#    Country -> Category -> App Type -> Rating Band
# ---------------------------------------------------------
hierarchy_levels = ["Country", "Display_Category", "App_Type", "Rating_Band"]
filtered_df["rating_x_reviews"] = filtered_df["Rating"] * filtered_df["Reviews"]

tree_records = []
parent_value_lookup = {}

for level_idx in range(1, len(hierarchy_levels) + 1):
  current_path = hierarchy_levels[:level_idx]
  parent_path = hierarchy_levels[: level_idx - 1]

  aggregated = filtered_df.groupby(current_path, as_index=False).agg(
      total_installs=("Installs", "sum"),
      total_reviews=("Reviews", "sum"),
      sum_weighted_rating=("rating_x_reviews", "sum"),
  )

  for _, row in aggregated.iterrows():
    node_id = " / ".join(str(row[c]) for c in current_path)
    node_label = str(row[current_path[-1]])

    if len(parent_path) == 0:
      parent_id = ""
      parent_val = row["total_installs"]
      pct_contribution = 100.0
    else:
      parent_id = " / ".join(str(row[c]) for c in parent_path)
      parent_val = parent_value_lookup.get(parent_id, row["total_installs"])
      pct_contribution = (
          (row["total_installs"] / parent_val * 100.0) if parent_val > 0 else 0.0
      )

    w_rating = (
        (row["sum_weighted_rating"] / row["total_reviews"])
        if row["total_reviews"] > 0
        else 0.0
    )
    is_highlighted = row["total_installs"] > 1_000_000

    tree_records.append({
        "id": node_id,
        "label": node_label,
        "parent": parent_id,
        "installs": row["total_installs"],
        "weighted_rating": w_rating,
        "reviews": row["total_reviews"],
        "pct_contribution": pct_contribution,
        "is_highlighted": is_highlighted,
    })
    parent_value_lookup[node_id] = row["total_installs"]

tree_df = pd.DataFrame(tree_records)

# ---------------------------------------------------------
# 6. Formatting Hover Information & Highlighting
# ---------------------------------------------------------
# Highlighting segments exceeding 1,000,000 installs
marker_line_colors = [
    "#FFD700" if h else "#FFFFFF" for h in tree_df["is_highlighted"]
]
marker_line_widths = [2.5 if h else 0.8 for h in tree_df["is_highlighted"]]

highlight_tags = [
    "<br><span style='color:gold;'>⭐ <b>High Volume:</b> Exceeds 1M"
    " Installs</span>"
    if h
    else ""
    for h in tree_df["is_highlighted"]
]

customdata_matrix = np.column_stack((
    tree_df["reviews"],
    tree_df["pct_contribution"],
    highlight_tags,
))

hovertemplate = (
    "<b>%{label}</b><br><br>"
    "<b>Total Installs:</b> %{value:,.0f}<br>"
    "<b>Weighted Rating:</b> %{color:.2f} ★<br>"
    "<b>Total Reviews:</b> %{customdata[0]:,.0f}<br>"
    "<b>Contribution to Parent:</b> %{customdata[1]:.2f}%"
    "%{customdata[2]}"
    "<extra></extra>"
)

# ---------------------------------------------------------
# 7. Construct Interactive Sunburst Figure
# ---------------------------------------------------------
fig = go.Figure(
    go.Sunburst(
        ids=tree_df["id"],
        labels=tree_df["label"],
        parents=tree_df["parent"],
        values=tree_df["installs"],
        branchvalues="total",
        marker=dict(
            colors=tree_df["weighted_rating"],
            colorscale="RdYlGn",
            cmin=4.0,
            cmax=5.0,
            colorbar=dict(
                title=dict(text="Weighted<br>Rating", side="top"),
                tickformat=".2f",
            ),
            line=dict(color=marker_line_colors, width=marker_line_widths),
        ),
        customdata=customdata_matrix,
        hovertemplate=hovertemplate,
        maxdepth=4,
    )
)

fig.update_layout(
    title=dict(
        text=(
            "<b>Global App Ecosystem Hierarchical Analysis</b><br>"
            "<sup>Structure: Country → Category → App Type → Rating Band | "
            "Size: Total Installs | Color: Weighted Rating</sup>"
        ),
        x=0.5,
    ),
    template="plotly_white",
    height=800,
    margin=dict(t=80, l=20, r=20, b=20),
)

# ---------------------------------------------------------
# 8. IST Time Restriction Gatekeeper (6:00 PM – 8:00 PM IST)
# ---------------------------------------------------------
current_ist = datetime.now(ZoneInfo("Asia/Kolkata"))
print(f"Current IST Time: {current_ist.strftime('%d %B %Y, %I:%M:%S %p')}")

# Active between 18:00:00 and 19:59:59 (6 PM to 8 PM IST)
if 18 <= current_ist.hour < 20:
  fig.show()
else:
  print(
      "Visualization unavailable.\n"
      "This visualization can only be viewed between 6:00 PM and 8:00 PM IST."
  )

Current IST Time: 24 September 2026, 07:13:18 PM
